# Импорты

In [ ]:
import os
import cv2
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Функция аугментации

In [ ]:
def apply_custom_augmentations(img, mask):

    img = img.copy()
    mask = mask.copy()


    # б) Случайное отражение по горизонтали
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
        mask = cv2.flip(mask, 1)

    # а) Случайный поворот (без обрезки краев, холст расширяется)
    if random.random() > 0.4:
        angle = random.uniform(-45, 45)
        h, w = img.shape[:2]
        cX, cY = w // 2, h // 2
        M = cv2.getRotationMatrix2D((cX, cY), angle, 1.0)

        # Пересчет новых размеров холста после поворота
        cos, sin = np.abs(M[0, 0]), np.abs(M[0, 1])
        nW = int((h * sin) + (w * cos))
        nH = int((h * cos) + (w * sin))

        # Корректируем матрицу сдвига
        M[0, 2] += (nW / 2) - cX
        M[1, 2] += (nH / 2) - cY

        img = cv2.warpAffine(img, M, (nW, nH), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        mask = cv2.warpAffine(mask, M, (nW, nH), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    # в) Случайный сдвиг
    if random.random() > 0.4:
        h, w = img.shape[:2]
        tx = random.randint(-int(w * 0.1), int(w * 0.1))
        ty = random.randint(-int(h * 0.1), int(h * 0.1))
        M = np.float32([[1, 0, tx], [0, 1, ty]])

        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    # г) Случайное масштабирование
    if random.random() > 0.4:
        scale = random.uniform(0.8, 1.2)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), 0, scale)

        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    # д) Случайный crop
    if random.random() > 0.5:
        h, w = img.shape[:2]
        crop_h = int(h * random.uniform(0.75, 0.95))
        crop_w = int(w * random.uniform(0.75, 0.95))

        start_x = random.randint(0, w - crop_w)
        start_y = random.randint(0, h - crop_h)

        img = img[start_y:start_y+crop_h, start_x:start_x+crop_w]
        mask = mask[start_y:start_y+crop_h, start_x:start_x+crop_w]

    # е) Случайный resize (изменение соотношения сторон)
    if random.random() > 0.5:
        h, w = img.shape[:2]
        new_w = int(w * random.uniform(0.8, 1.2))
        new_h = int(h * random.uniform(0.8, 1.2))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    # ё) Перспективное искажение
    if random.random() > 0.4:
        h, w = img.shape[:2]
        pts1 = np.float32([[0, 0], [w - 1, 0], [0, h - 1], [w - 1, h - 1]])

        dx, dy = w * 0.07, h * 0.07
        pts2 = np.float32([
            [random.uniform(0, dx), random.uniform(0, dy)],
            [w - 1 - random.uniform(0, dx), random.uniform(0, dy)],
            [random.uniform(0, dx), h - 1 - random.uniform(0, dy)],
            [w - 1 - random.uniform(0, dx), h - 1 - random.uniform(0, dy)]
        ])

        P = cv2.getPerspectiveTransform(pts1, pts2)
        img = cv2.warpPerspective(img, P, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        mask = cv2.warpPerspective(mask, P, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    # ж) Небольшая деформация (Elastic Transform)
    if random.random() > 0.4:
        h, w = img.shape[:2]
        dx = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigmaX=15, sigmaY=15) * 6
        dy = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigmaX=15, sigmaY=15) * 6

        x, y = np.meshgrid(np.arange(w), np.arange(h))
        map_x = (x + dx).astype(np.float32)
        map_y = (y + dy).astype(np.float32)

        img = cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        mask = cv2.remap(mask, map_x, map_y, interpolation=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    # з) Изменение яркости и и) Изменение контраста
    if random.random() > 0.4:
        contrast = random.uniform(0.85, 1.15)
        brightness = random.randint(-20, 20)
        img = cv2.convertScaleAbs(img, alpha=contrast, beta=brightness)

    # й) Изменение насыщенности и к) Изменение оттенка
    if random.random() > 0.4:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = (hsv[:, :, 0] + random.randint(-12, 12)) % 180
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.8, 1.2), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # л) Изменение температуры цвета
    if random.random() > 0.4:
        temp_shift = random.randint(-15, 15)
        img = img.astype(np.int16)
        if temp_shift > 0:
            img[:, :, 0] = np.clip(img[:, :, 0] + temp_shift, 0, 255)
            img[:, :, 2] = np.clip(img[:, :, 2] - temp_shift // 2, 0, 255)
        else:
            img[:, :, 2] = np.clip(img[:, :, 2] - temp_shift, 0, 255)
            img[:, :, 0] = np.clip(img[:, :, 0] + temp_shift // 2, 0, 255)
        img = img.astype(np.uint8)

    # м) Случайная гамма-коррекция
    if random.random() > 0.4:
        gamma = random.uniform(0.8, 1.2)
        invGamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        img = cv2.LUT(img, table)

    # н) Случайный шум
    if random.random() > 0.4:
        noise = np.random.normal(0, random.uniform(4, 12), img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # Приведение к фиксированному размеру под сеть 512x512
    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)

    return img, mask

# Класс Dataset для PyTorch

1. PyTorch не умеет сам ходить по  папкам, открывать картинки, приводить их к нужному размеру и превращать в тензоры.
2. Класс NailDataset стандартизирует этот процесс

In [ ]:
# класс датасета, наследуясь от базового Dataset в PyTorch
class NailDataset(Dataset):

    # Конструктор класса. Принимает пути к папкам с картинками, масками и флаг режима обучения
    def __init__(self, images_dir, masks_dir, is_train=True):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.is_train = is_train      # True (обучение) или False (валидация/тест)

        # Читаем имена всех файлов в папке картинок и сортируем их, чтобы порядок всегда совпадал с масками
        self.image_names = sorted(os.listdir(images_dir))

    #  метод, который возвращает общее количество картинок в датасете
    def __len__(self):
        return len(self.image_names)  # PyTorch вызывает его, чтобы знать, когда заканчивается эпоха

    # Возвращает ОДНУ пару (картинка, маска) по указанному индексу idx
    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.images_dir, img_name)

        mask_name = os.path.splitext(img_name)[0] + '.jpg'
        mask_path = os.path.join(self.masks_dir, mask_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)

        if self.is_train:
            image, mask = apply_custom_augmentations(image, mask) # если тренеруем
        else:
            # Если мы тестируем
            image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)


        # Нормализация: переводим пиксели фото во float32 и делим на 255.0, получая значения от 0.0 до 1.0
        image = image.astype(np.float32) / 255.0

        # Транспонирование осей: NumPy хранит картинки как (Высота, Ширина, Каналы), а PyTorch требует (Каналы, Высота, Ширина)
        image = image.transpose(2, 0, 1)

        mask = np.expand_dims(mask, axis=0).astype(np.float32)

        # Превращаем готовые массивы NumPy в тензоры PyTorch и возвращаем их
        return torch.tensor(image), torch.tensor(mask)

# Архитектура модели UNET

1. Conv2d (Свёртка): Сканирует картинку маленьким «окошком» $3 \times 3$, чтобы находить границы, текстуру и изгибы ногтя.BatchNorm2d 
2. (Батч-нормализация): Масштабирует числа внутри сети, чтобы обучение  шло стабильно. 
3. ReLU: Функция активации, которая зануляет отрицательные значения, добавляя сети «нелинейности» (чтобы она могла учить сложные зависимости, а не просто складывать пиксели).

In [ ]:
# Создаем класс для повторяющегося блока из двух сверток подряд
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            # 1-я свёртка: ищет паттерны, ядро 3х3, padding=1 сохраняет размер картинки (рамка). bias=False, так как дальше идет BatchNorm
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            # 1-я нормализация: стабилизирует промежуточные числа, ускоряет обучение
            nn.BatchNorm2d(out_channels),
            # 1-я активация ReLU: зануляет отрицательные значения, добавляя сети нелинейность (inplace экономит память)
            nn.ReLU(inplace=True),

            # 2-я свёртка: углубляет обработку признаков (на входе и выходе уже одинаковое число каналов)
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            # 2-я нормализация: выравнивает масштабы данных после второй свертки
            nn.BatchNorm2d(out_channels),
            # 2-я активация ReLU: окончательно формирует карту признаков этого блока
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x) # Просто пускаем тензор x через наш конвейер сверток

Unet

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__() # Инициализируем базовый класс

        #  ЭНКОДЕР (Левая сторона "U", сжатие картинки)
        # превращаем 3 канала цвета в 64 канала признаков далее анлогично
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        # уменьшает высоту и ширину картинки ровно в 2 раза, отбрасывая менее важные пиксели
        self.pool = nn.MaxPool2d(2)

        # Самый глубокий слой: принимает 512 каналов, выдает 1024. Картинка тут самая маленькая
        self.bottleneck = DoubleConv(512, 1024)

        # ДЕКОДЕР (Правая сторона "U", восстановление размера)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up_conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        # Обработка после склейки: на входе 512 (256 + 256), на выходе 256
        self.up_conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv1 = DoubleConv(128, 64)

        # Финальная свертка 1х1: сжимает 64 канала признаков в итоговое количество каналов маски (out_channels=1)
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)


    # ПРЯМОЙ ПРОХОД сам процесс обучения (Маршрут движения данных через сеть при инференсе/обучении)
    def forward(self, x):
        #  Прогоняем через первый блок сверток (получаем x1, размер исходный, например, 512х512)
        x1 = self.down1(x)
        # Уменьшаем x1 в два раза через пулинг и прогоняем через второй блок сверток (получаем x2)
        x2 = self.down2(self.pool(x1))
        #  Снова уменьшаем размер в два раза и прогоняем через третий блок сверток (получаем x3)
        x3 = self.down3(self.pool(x2))
        #  Еще раз уменьшаем размер и берем четвертый блок сверток (получаем x4)
        x4 = self.down4(self.pool(x3))

        # Последний раз уменьшаем x4 и прогоняем через "дно" сети (получаем b — bottleneck)
        b = self.bottleneck(self.pool(x4))

        # Начинаем подъем вверх (Декодер)
        #  Апсемплинг (растягивание) нижнего тензора b
        x = self.up4(b)
        # Склеиваем по оси каналов (dim=1) растянутый x и точный x4 из левой части (Skip Connection)
        x = torch.cat([x, x4], dim=1)
        #  Сглаживаем склеенный "бутерброд" двойной сверткой
        x = self.up_conv4(x)

        #  Растягиваем тензор x вверх
        x = self.up3(x)
        #  Склеиваем полученный результат с точным x3 из левой части (Skip Connection)
        x = torch.cat([x, x3], dim=1)
        #  Сглаживаем склеенные данные двойной сверткой
        x = self.up_conv3(x)

        #  Растягиваем тензор x вверх
        x = self.up2(x)
        # Склеиваем с точным x2 из левой части (Skip Connection)
        x = torch.cat([x, x2], dim=1)
        #  Сглаживаем данные двойной сверткой
        x = self.up_conv2(x)

        #  Растягиваем тензор x вверх до первоначального размера картинки
        x = self.up1(x)
        #  Склеиваем с самым первым, самым детальным слоем x1 (Skip Connection)
        x = torch.cat([x, x1], dim=1)
        #  Сглаживаем финальный глубокий набор признаков двойной сверткой
        x = self.up_conv1(x)

        # Схлопываем 64 канала в 1 канал маски с помощью свёртки 1х1 и возвращаем результат
        return self.out_conv(x)

# Гиперпараметры и цикл обучения

In [ ]:
TRAIN_IMG_DIR = "LAB_6/Nails/nails_segmentation/images"
TRAIN_MASK_DIR = "LAB_6/Nails/nails_segmentation/labels"


DEVICE = torch.device("cpu")
# сколько картинок нейросеть будет смотреть одновременно за один шаг
BATCH_SIZE = 4
# Количество циклов обучения
EPOCHS = 20
#  шаг, с которым оптимизатор будет корректировать веса сети
LR = 1e-4

# Инициализация пайплайна данных
# Создаю объект датасета, передаем ему пути к папкам и включаем режим обучения
train_dataset = NailDataset(images_dir=TRAIN_IMG_DIR, masks_dir=TRAIN_MASK_DIR, is_train=True)
# Создаю DataLoader: он перемешивает данные, собирает их в батчи по 4 шт, грузит в 2 потока и отбрасывает неполный последний батч
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

# Создаю экземпляр  модели U-Net
model = UNet(in_channels=3, out_channels=1).to(DEVICE)
# Задаю функцию потерь
criterion = nn.BCEWithLogitsLoss() # Бинарная кросс-энтропия с логитами так как маска бинарная (0,1)
# Задаю оптимизатор он будет крутить веса модели (model.parameters) на основе градиентов со скоростью LR
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Старт обучения на устройстве: {DEVICE}")
print(f"Найдено изображений: {len(train_dataset)}")

for epoch in range(EPOCHS):
    # Переводим модель в режим обучения (активирует BatchNorm и включает подсчет градиентов)
    model.train()
    # Обнуляем накопитель ошибки для текущей эпохи
    epoch_loss = 0

    # Внутренний цикл: идем по батчам. batch_idx — номер шага, images — 4 картинки, masks — 4 маски
    for batch_idx, (images, masks) in enumerate(train_loader):
        # Переношу батч с картинками на выбранное устройство
        images = images.to(DEVICE)
        # Переношу батч с правильными масками на то же устройство
        masks = masks.to(DEVICE)

        # ПРЯМОЙ ПРОХОД: отдаю картинки модели, запускается метод forward, на выходе получаю предсказания
        outputs = model(images)
        # сравниваю то, что угадала нейросеть , с тем, как должно быть на самом деле
        loss = criterion(outputs, masks)

        #  очищаю данные о наклоне функции ошибки с прошлого шага, чтобы они не складывались
        optimizer.zero_grad()
        # вычисляю, насколько сильно ошибся каждый конкретный весовой коэффициент сети
        loss.backward()
        #  физически обновляю  веса нейросети в сторону уменьшения ошибки
        optimizer.step()

        # Достаю числовое значение ошибки из тензора PyTorch и прибавляем его к сумме за эту эпоху
        epoch_loss += loss.item()

    # Считаю среднюю ошибку за всю эпоху
    avg_loss = epoch_loss / len(train_loader)
    print(f"Эпоха [{epoch+1}/{EPOCHS}] | Средний Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), "unet_nails_model.pth")